In [1]:
# Setup: load normalized products, candidate pairs (from explore.ipynb) and labels (from labeler/)
import json
from datetime import datetime
from difflib import SequenceMatcher
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from xgboost import XGBClassifier

products = pd.read_parquet("../data/products_normalized.parquet")
candidate_pairs = pd.read_parquet("../data/candidate_pairs.parquet")

# dtype=str: otherwise pandas turns EAN codes into integers and drops leading zeros
labels = pd.read_json("../labels/labels.jsonl", lines=True, dtype={"coop_key": str})
labels = labels.drop_duplicates("coop_key", keep="last")  # latest answer wins if you relabeled

# The held-out test set (labels/split.json) stays out of everything here, it's only scored in match.ipynb
split = json.loads(Path("../labels/split.json").read_text())
labels = labels[~labels["order"].between(split["test_from_order"], split["test_to_order"] - 1)]
print(labels["status"].value_counts())

status
match    171
none     110
skip      20
Name: count, dtype: int64


In [2]:
# One row per (Coop product, shown ICA candidate), y = 1 if you selected it. Skipped products are left out.
labeled = labels[labels["status"].isin(["match", "none"])]
pairs = (
    labeled[["coop_key", "shown", "matches"]]
    .explode("shown")
    .rename(columns={"shown": "ica_key"})
    .merge(candidate_pairs[["coop_key", "ica_key", "rank", "score"]], on=["coop_key", "ica_key"], how="left")
)
pairs["y"] = [ica in matches for ica, matches in zip(pairs["ica_key"], pairs["matches"])]
pairs["y"] = pairs["y"].astype(int)
pairs = pairs.drop(columns="matches")

# If candidate_pairs was regenerated after labeling, some shown pairs may be missing a score
assert pairs["score"].notna().all(), "candidate_pairs.parquet changed since labeling"
print(f"{len(pairs):,} pairs, {pairs['y'].sum()} positives, {pairs['coop_key'].nunique()} Coop products")

2,810 pairs, 175 positives, 281 Coop products


In [3]:
# Pair features
INFO = ["product_key", "name", "brand_norm", "name_norm", "size_value", "size_unit", "price"]
coop_info = products[products["store"] == "coop"][INFO].add_prefix("coop_")
ica_info = products[products["store"] == "ica"][INFO].add_prefix("ica_")

def similarity(a, b):
    return SequenceMatcher(None, a, b).ratio()

def jaccard(a, b):
    a, b = set(a.split()), set(b.split())
    return len(a & b) / len(a | b) if a | b else 0.0

def add_features(df):
    df = (
        df.merge(coop_info, left_on="coop_key", right_on="coop_product_key")
          .merge(ica_info, left_on="ica_key", right_on="ica_product_key")
    )
    # How this candidate compares to the best TF-IDF candidate for the same Coop product
    df["score_gap"] = df.groupby("coop_key")["score"].transform("max") - df["score"]

    df["brand_equal"] = (df["coop_brand_norm"] == df["ica_brand_norm"]).astype(int)
    df["brand_sim"] = [similarity(a, b) for a, b in zip(df["coop_brand_norm"], df["ica_brand_norm"])]
    df["name_sim"] = [similarity(a, b) for a, b in zip(df["coop_name_norm"], df["ica_name_norm"])]
    df["name_jaccard"] = [jaccard(a, b) for a, b in zip(df["coop_name_norm"], df["ica_name_norm"])]

    # Size and price: NaN when unknown, xgboost handles missing values itself
    same_unit = df["coop_size_unit"] == df["ica_size_unit"]
    df["same_unit"] = np.where(df["coop_size_unit"].isna() | df["ica_size_unit"].isna(), np.nan, same_unit)
    sizes = df[["coop_size_value", "ica_size_value"]].where(same_unit, axis=0)
    df["size_ratio"] = sizes.min(axis=1) / sizes.max(axis=1)
    prices = df[["coop_price", "ica_price"]]
    df["price_ratio"] = prices.min(axis=1) / prices.max(axis=1)
    return df

FEATURES = [
    "score", "rank", "score_gap",
    "brand_equal", "brand_sim", "name_sim", "name_jaccard",
    "same_unit", "size_ratio", "price_ratio",
]

pairs = add_features(pairs)
pairs[FEATURES].describe().T[["mean", "min", "max"]]

,mean,min,max
score,0.543028,0.158151,1.000000
rank,5.500000,1.000000,10.000000
score_gap,0.217500,0.000000,0.770881
brand_equal,0.636299,0.000000,1.000000
brand_sim,0.741745,0.000000,1.000000
name_sim,0.523698,0.000000,1.000000
name_jaccard,0.234351,0.000000,1.000000
same_unit,0.821458,0.000000,1.000000
size_ratio,0.680055,0.001000,1.000000
price_ratio,0.643748,0.003480,1.000000


In [4]:
# Product-level evaluation: per Coop product, take the best candidate if its score >= threshold, else "none".
# precision = predicted matches that are right, recall = real matches found,
# accuracy = products fully right (a "none" product counts as right when nothing is predicted)
def evaluate(df, score_col, threshold):
    # ties (e.g. ICA duplicates with the same score) go to the better TF-IDF rank
    best = df.sort_values([score_col, "rank"], ascending=[False, True]).groupby("coop_key").head(1)
    has_match = df.groupby("coop_key")["y"].max()
    predicted = best[best[score_col] >= threshold]

    correct = predicted["y"].sum()
    n_predicted, n_real = len(predicted), has_match.sum()
    n_correct_none = ((has_match == 0) & ~has_match.index.isin(predicted["coop_key"])).sum()
    return {
        "precision": correct / n_predicted if n_predicted else 0.0,
        "recall": correct / n_real,
        "accuracy": (correct + n_correct_none) / len(has_match),
        "threshold": threshold,
    }

def best_threshold(df, score_col, metric="accuracy"):
    results = [evaluate(df, score_col, t) for t in np.arange(0, 1.0, 0.05)]
    return max(results, key=lambda r: r[metric])

In [5]:
# Experiment log: one row per run in ../experiments.csv, so you can follow the improvements over time
LOG_PATH = Path("../experiments.csv")

def log_experiment(name, metrics, notes=""):
    row = {
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "name": name,
        "n_products": pairs["coop_key"].nunique(),
        "n_matches": int(pairs["y"].sum()),
        **{k: round(float(v), 3) for k, v in metrics.items()},
        "features": ",".join(FEATURES) if name != "baseline_tfidf" else "",
        "notes": notes,
    }
    log = pd.read_csv(LOG_PATH) if LOG_PATH.exists() else pd.DataFrame()
    # Don't log the same result twice when rerunning the notebook
    same = ["name", "n_products", "precision", "recall", "accuracy"]
    if not log.empty and (log[same] == pd.Series(row)[same]).all(axis=1).any():
        print("already logged")
    else:
        pd.concat([log, pd.DataFrame([row])], ignore_index=True).to_csv(LOG_PATH, index=False)
    return pd.read_csv(LOG_PATH)

In [6]:
# Baseline which is just the TF-IDF score from candidate generation.
# The threshold is tuned on the same data, so this is slightly optimistic.
always_rank1 = evaluate(pairs, "score", threshold=0)
baseline = best_threshold(pairs, "score")
print("always rank 1:     ", {k: round(v, 3) for k, v in always_rank1.items()})
print("tfidf + threshold: ", {k: round(v, 3) for k, v in baseline.items()})
log_experiment("baseline_tfidf", baseline, notes="TF-IDF score only, threshold tuned on all labels")

always rank 1:      {'precision': np.float64(0.459), 'recall': np.float64(0.754), 'accuracy': np.float64(0.459), 'threshold': 0}
tfidf + threshold:  {'precision': np.float64(0.667), 'recall': np.float64(0.702), 'accuracy': np.float64(0.726), 'threshold': np.float64(0.7)}


,timestamp,name,n_products,n_matches,precision,recall,accuracy,threshold,features,notes
0,2026-10-21T18:52:10,baseline_tfidf,147,93,0.742,0.742,0.776,0.7,NaN,"TF-IDF score only, threshold tuned on all labels"
1,2026-10-21T18:52:11,xgboost_v1,147,93,0.939,0.828,0.871,0.5,"score,rank,score_gap,brand_equal,brand_sim,nam...",first model
2,2026-10-22T15:05:48,baseline_tfidf,231,142,0.699,0.723,0.749,0.7,NaN,"TF-IDF score only, threshold tuned on all labels"
3,2026-10-22T15:05:49,xgboost_v1,231,142,0.933,0.787,0.844,0.5,"score,rank,score_gap,brand_equal,brand_sim,nam...",first model
4,2026-10-22T15:37:17,baseline_tfidf,281,175,0.667,0.702,0.726,0.7,NaN,"TF-IDF score only, threshold tuned on all labels"


In [7]:
# Model: xgboost on the pair features, with 5-fold cross-validation grouped by Coop product
# (all candidates of a product stay in the same fold, so the model never sees the product it's tested on)
def make_model():
    return XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.05, subsample=0.8, random_state=0)

pairs["prob"] = np.nan
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0)
for train_idx, test_idx in cv.split(pairs, pairs["y"], groups=pairs["coop_key"]):
    model = make_model().fit(pairs.iloc[train_idx][FEATURES], pairs.iloc[train_idx]["y"])
    pairs.loc[pairs.index[test_idx], "prob"] = model.predict_proba(pairs.iloc[test_idx][FEATURES])[:, 1]

result = evaluate(pairs, "prob", threshold=0.5)
print("xgboost @ 0.5:", {k: round(v, 3) for k, v in result.items()})
log_experiment("xgboost_v1", result, notes="first model")

xgboost @ 0.5: {'precision': np.float64(0.889), 'recall': np.float64(0.749), 'accuracy': np.float64(0.815), 'threshold': 0.5}


,timestamp,name,n_products,n_matches,precision,recall,accuracy,threshold,features,notes
0,2026-10-21T18:52:10,baseline_tfidf,147,93,0.742,0.742,0.776,0.7,NaN,"TF-IDF score only, threshold tuned on all labels"
1,2026-10-21T18:52:11,xgboost_v1,147,93,0.939,0.828,0.871,0.5,"score,rank,score_gap,brand_equal,brand_sim,nam...",first model
2,2026-10-22T15:05:48,baseline_tfidf,231,142,0.699,0.723,0.749,0.7,NaN,"TF-IDF score only, threshold tuned on all labels"
3,2026-10-22T15:05:49,xgboost_v1,231,142,0.933,0.787,0.844,0.5,"score,rank,score_gap,brand_equal,brand_sim,nam...",first model
4,2026-10-22T15:37:17,baseline_tfidf,281,175,0.667,0.702,0.726,0.7,NaN,"TF-IDF score only, threshold tuned on all labels"
5,2026-10-22T15:37:18,xgboost_v1,281,175,0.889,0.749,0.815,0.5,"score,rank,score_gap,brand_equal,brand_sim,nam...",first model


In [8]:
# Precision / recall at different thresholds: where would you put the cut-off?
pd.DataFrame([evaluate(pairs, "prob", t) for t in np.arange(0.1, 1.0, 0.1)]).round(3)

,precision,recall,accuracy,threshold
0,0.791,0.883,0.840,0.1
1,0.826,0.860,0.854,0.2
2,0.847,0.842,0.861,0.3
3,0.864,0.778,0.829,0.4
4,0.889,0.749,0.815,0.5
5,0.899,0.731,0.804,0.6
6,0.905,0.667,0.772,0.7
7,0.945,0.602,0.751,0.8
8,0.957,0.515,0.701,0.9


In [9]:
# What the model relies on (trained on all labels)
model = make_model().fit(pairs[FEATURES], pairs["y"])
pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False).round(3)

score_gap       0.361
size_ratio      0.242
score           0.100
rank            0.087
price_ratio     0.068
brand_equal     0.034
name_jaccard    0.032
name_sim        0.028
same_unit       0.026
brand_sim       0.022
dtype: float32

In [10]:
# Errors: the products where the model's answer (threshold 0.5) differs from your label.
# This is where to look for new features, normalization bugs or labeling mistakes.
# A product can have several accepted ICA listings (duplicates), predicting any of them counts as right.
THRESHOLD = 0.5
best = pairs.sort_values(["prob", "rank"], ascending=[False, True]).groupby("coop_key").head(1).set_index("coop_key")
truth = pairs[pairs["y"] == 1].groupby("coop_key").agg(keys=("ica_key", set), names=("ica_name", " / ".join))

errors = []
for coop_key, b in best.iterrows():
    predicted_key = b["ica_key"] if b["prob"] >= THRESHOLD else None
    actual_keys = truth.loc[coop_key, "keys"] if coop_key in truth.index else set()
    is_correct = predicted_key in actual_keys if predicted_key else not actual_keys
    if not is_correct:
        errors.append({
            "coop": b["coop_name"],
            "predicted": b["ica_name"] if predicted_key else "(none)",
            "prob": round(b["prob"], 2),
            "label": truth.loc[coop_key, "names"] if actual_keys else "(none)",
        })

pd.set_option("display.max_colwidth", 60)
print(f"{len(errors)} of {len(best)} products wrong")
pd.DataFrame(errors)

52 of 281 products wrong


,coop,predicted,prob,label
0,Winston Green 100s,Green 100s 20-p Winston,0.99,Winston Green 100s 20-p
1,Kattmat Classics Kyckling,Kattmat Kyckling 85g Sheba,0.96,Kattmat Kyckling 85g Sheba
2,Glutenfria Korvbröd 5-pack,Korvbröd Glutenfria 5-p,0.92,(none)
3,Kaffesirap Vanilj,Kaffesirap Caramel 375ml Saturnus 1893,0.91,"Vanilla Kaffesmaksättare 37,5cl Saturnus 1893"
4,Jumperstickor Aluminium 3mm,Jumperstickor Aluminium 5mm 1st Falk,0.89,"Jumperstickor 3,0 MM 1par Falk"
5,Skrivbok A5,Skrivbok A5 1st Writer,0.87,(none)
6,General Slim Stock,Slim White Stock General,0.76,(none)
7,Foul Favabönor,Favabönor Foul Libanesisk 400g Al Wadi,0.74,(none)
8,Kaffe Djäkneberget Hela Bönor,Kaffe Svartån Eko Hela Bönor 500g Björklunds Kafferosteri,0.73,(none)
9,Sriracha Mayo Sauce,Sriracha Chilisås 455ml Flying Goose,0.73,Majonnäs Sriracha 455ml Flying Goose
